# LeetCode 2155 — All Divisions With the Highest Score of a Binary Array bằng Python

Link đề: https://leetcode.com/problems/all-divisions-with-the-highest-score-of-a-binary-array/ · Độ khó: **Medium** · Chủ đề: Array

Skeleton LeetCode:

```python
class Solution:
    def maxScoreIndices(self, nums: List[int]) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Hiểu **n + 1** điểm chia `i = 0..n` (kể cả hai đầu với một phía rỗng) và công thức điểm `score(i) = (#0 trong nums[0..i−1]) + (#1 trong nums[i..n−1])`.
2. Cập nhật điểm **O(1)** khi dịch điểm chia sang phải: phần tử `nums[i]` đi từ phải sang trái ⇒ `+1` nếu nó là 0, `−1` nếu nó là 1.
3. Thu thập **mọi chỉ số đạt max** trong một lượt: gặp max mới thì **reset** danh sách, gặp bằng max thì **thêm**.
4. So sánh: vét cạn O(n²), tiền tố hai mảng, một lượt O(1) bộ nhớ, và **biến đổi về "max prefix sum"** (điểm = hằng số + tổng tiền tố của `±1`).
5. Test với đáp án "thứ tự tùy ý" và các edge case toàn 0 / toàn 1.

## 2. Đề bài & ví dụ

Mảng nhị phân `nums` độ dài `n` có thể chia tại chỉ số `i` (`0 ≤ i ≤ n`) thành hai mảng (có thể rỗng):
- `numsleft = nums[0..i−1]`, `numsright = nums[i..n−1]`.
- `i = 0`: trái rỗng; `i = n`: phải rỗng.

**Điểm** của chỉ số `i` = số **0** trong `numsleft` + số **1** trong `numsright`.

Trả về **mọi** chỉ số `i` có điểm **cao nhất** (thứ tự tùy ý).

**Ví dụ 1:** `nums = [0,0,1,0]` → `[2,4]` (điểm lần lượt 1, 2, 3, 2, 3).

**Ví dụ 2:** `nums = [0,0,0]` → `[3]` (điểm 0, 1, 2, 3).

**Ví dụ 3:** `nums = [1,1]` → `[0]` (điểm 2, 1, 0).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `nums[i] ∈ {0, 1}`.

In [1]:
from typing import List
from itertools import accumulate
import random, time

examples = [([0, 0, 1, 0], [2, 4]), ([0, 0, 0], [3]), ([1, 1], [0])]
def score(nums, i):
    return nums[:i].count(0) + nums[i:].count(1)

for nums, exp in examples:
    print(nums, "điểm theo i:", [score(nums, i) for i in range(len(nums) + 1)], "-> kỳ vọng", exp)

[0, 0, 1, 0] điểm theo i: [1, 2, 3, 2, 3] -> kỳ vọng [2, 4]
[0, 0, 0] điểm theo i: [0, 1, 2, 3] -> kỳ vọng [3]
[1, 1] điểm theo i: [2, 1, 0] -> kỳ vọng [0]


## 3. Kiến thức nền

### 3.1 Lát cắt rỗng và `count`

`nums[:0]` và `nums[n:]` là list rỗng — `count` trả 0, không lỗi. Nhờ vậy công thức `score(i)` đúng cả với `i = 0` và `i = n`.

In [2]:
a = [0, 0, 1, 0]
print(a[:0], a[4:], a[:0].count(0), a[4:].count(1))

[] [] 0 0


### 3.2 `enumerate(nums, 1)` — chỉ số bắt đầu từ 1

Khi duyệt phần tử `nums[i−1]` mà muốn biết "điểm chia ngay sau nó" là `i`, dùng `enumerate(nums, 1)`.

In [3]:
for i, x in enumerate([0, 0, 1, 0], 1):
    print(f"sau khi chuyển nums[{i-1}]={x} sang trái, điểm chia là i={i}")

sau khi chuyển nums[0]=0 sang trái, điểm chia là i=1
sau khi chuyển nums[1]=0 sang trái, điểm chia là i=2
sau khi chuyển nums[2]=1 sang trái, điểm chia là i=3
sau khi chuyển nums[3]=0 sang trái, điểm chia là i=4


### 3.3 Mẹo `x ^ 1` và `1 - x` với bit

Với `x ∈ {0, 1}`: `x ^ 1 = 1 − x` = "x có phải là 0 không".

In [4]:
print([(x, x ^ 1, 1 - x) for x in (0, 1)])

[(0, 1, 1), (1, 0, 0)]


### 3.4 Tổng tiền tố để đếm nhanh

`pre0[i]` = số 0 trong `nums[0..i−1]`; số 1 trong `nums[i..n−1]` = `tổng1 − pre1[i]`. Mỗi `score(i)` tính O(1) sau khi có tiền tố.

In [5]:
a = [0, 0, 1, 0]
pre0 = list(accumulate((x ^ 1 for x in a), initial=0))
pre1 = list(accumulate(a, initial=0))
print("pre0 =", pre0, "pre1 =", pre1, "tổng1 =", pre1[-1])
print([pre0[i] + pre1[-1] - pre1[i] for i in range(len(a) + 1)])

pre0 = [0, 1, 2, 2, 3] pre1 = [0, 0, 0, 1, 1] tổng1 = 1
[1, 2, 3, 2, 3]


## 4. Ý tưởng nền tảng / trực giác

Hình dung một "vách ngăn" di chuyển từ trái sang phải qua `n + 1` vị trí. Bên trái vách ta **thích số 0**, bên phải vách ta **thích số 1**.

Khi vách nhảy qua phần tử `nums[i]` (từ vị trí `i` sang `i + 1`), phần tử này chuyển từ **phải** sang **trái**:
- Nếu nó là **0**: trước đó nó không được tính (bên phải chỉ tính số 1), giờ được tính (bên trái tính số 0) ⇒ điểm **+1**.
- Nếu nó là **1**: trước đó được tính, giờ không ⇒ điểm **−1**.

Điểm ban đầu `score(0)` = tổng số 1 (trái rỗng, phải là cả mảng).

Tính tay ví dụ 1 `[0, 0, 1, 0]`, tổng số 1 = 1:

| i | phần tử vừa nhảy qua | thay đổi | score(i) |
|---|---|---|---|
| 0 | — | — | 1 |
| 1 | 0 | +1 | 2 |
| 2 | 0 | +1 | 3 |
| 3 | 1 | −1 | 2 |
| 4 | 0 | +1 | 3 |

Max = 3 tại `i = 2` và `i = 4`. ✔

**Ví dụ 3** `[1, 1]`: `score(0) = 2`; qua 1 ⇒ 1; qua 1 ⇒ 0. Max tại `i = 0` — vách ngăn nên đứng **trước** mọi số 1. Tổng quát: điểm cao khi bên trái "nhiều 0" và bên phải "nhiều 1", nên vách thường dừng ở **ranh giới** "0 hết, 1 bắt đầu".

**Hình dung "đường đi bộ":** vẽ `score(i)` theo `i` — đó là một đường gấp khúc đi lên 1 khi gặp số 0, đi xuống 1 khi gặp số 1. Ta cần mọi **đỉnh cao nhất** của đường gấp khúc này.

In [6]:
def ascii_walk(nums):
    sc = [sum(nums)]
    for x in nums: sc.append(sc[-1] + (1 if x == 0 else -1))
    lo, hi = min(sc), max(sc)
    for level in range(hi, lo - 1, -1):
        print(f"{level:>3} | " + " ".join("*" if v == level else " " for v in sc))
    print("    +-" + "--" * len(sc)); print("  i:  " + " ".join(str(i % 10) for i in range(len(sc))))

ascii_walk([1, 0, 0, 1, 0, 0, 1, 1])

  6 |             *    
  5 |       *   *   *  
  4 | *   *   *       *
  3 |   *              
    +-------------------
  i:  0 1 2 3 4 5 6 7 8


In [7]:
nums = [0, 0, 1, 0]
s = sum(nums); print(f"i=0: score={s}")
for i, x in enumerate(nums, 1):
    s += 1 if x == 0 else -1
    print(f"i={i}: nhảy qua {x} -> {'+1' if x == 0 else '-1'} -> score={s}")

i=0: score=1
i=1: nhảy qua 0 -> +1 -> score=2
i=2: nhảy qua 0 -> +1 -> score=3
i=3: nhảy qua 1 -> -1 -> score=2
i=4: nhảy qua 0 -> +1 -> score=3


## 5. Cách 1 — Brute Force / baseline: tính lại điểm cho từng `i`

In [8]:
def brute(nums):
    n = len(nums)
    scores = [nums[:i].count(0) + nums[i:].count(1) for i in range(n + 1)]
    mx = max(scores)
    return [i for i, s in enumerate(scores) if s == mx]

for nums, exp in examples:
    got = brute(nums)
    print(got, "OK" if sorted(got) == sorted(exp) else "SAI")

[2, 4] OK
[3] OK
[0] OK


**Độ phức tạp:** mỗi `score(i)` tốn O(n) (cắt lát + đếm) ⇒ **O(n²)**; `n = 10⁵` ⇒ 10¹⁰ phép.

**Nút thắt:** `score(i)` và `score(i + 1)` chỉ khác nhau ở **một** phần tử (`nums[i]`), nhưng ta đếm lại từ đầu.

In [9]:
for n in [2000, 5000]:
    a = [random.randint(0, 1) for _ in range(n)]
    t = time.perf_counter(); brute(a); print(f"n={n}: {(time.perf_counter()-t)*1000:.0f} ms")

n=2000: 28 ms


n=5000: 149 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (cập nhật O(1)):** `score(i + 1) = score(i) + (1 nếu nums[i] = 0, ngược lại −1)`, `score(0) = Σ nums`.

**Insight 2 (gom chỉ số max trong một lượt):** duy trì `mx` và danh sách `ans`:
- **Nếu** `score > mx` **thì** `mx = score`, `ans = [i]` (bỏ hết chỉ số cũ).
- **Nếu** `score == mx` **thì** `ans.append(i)`.
- **Ngược lại** bỏ qua.

**Insight 3 (biến đổi thành max prefix sum):** đặt `w = +1` cho số 0, `−1` cho số 1. Khi đó `score(i) = Σnums + (w_0 + … + w_{i−1})` = hằng số + tổng tiền tố `W[i]`. Vậy các chỉ số max của `score` = các chỉ số max của **tổng tiền tố** `W` — một bài kinh điển.

**Bất biến khi quét:** sau khi xử lý điểm chia `i`, `mx = max(score(0..i))` và `ans` = mọi `j ≤ i` có `score(j) = mx`, theo thứ tự tăng.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tiền tố hai mảng, rồi tìm max

In [10]:
def prefix_arrays(nums):
    n = len(nums)
    pre0 = list(accumulate((x ^ 1 for x in nums), initial=0))
    pre1 = list(accumulate(nums, initial=0))
    total1 = pre1[-1]
    scores = [pre0[i] + total1 - pre1[i] for i in range(n + 1)]
    mx = max(scores)
    return [i for i, s in enumerate(scores) if s == mx]

for nums, exp in examples:
    got = prefix_arrays(nums)
    print(got, "OK" if sorted(got) == sorted(exp) else "SAI")

[2, 4] OK
[3] OK
[0] OK


**Độ phức tạp cách 2:** O(n) thời gian, O(n) bộ nhớ (ba mảng). Dễ kiểm tra từng bước.

### 7.2 Cách 3 — Một lượt, cập nhật O(1), gom chỉ số max (cách doocs)

In [11]:
def one_pass(nums):
    score = sum(nums)                 # i = 0
    mx = score; ans = [0]
    for i, x in enumerate(nums, 1):
        score += 1 if x == 0 else -1
        if score > mx:
            mx = score; ans = [i]
        elif score == mx:
            ans.append(i)
    return ans

for nums, exp in examples:
    got = one_pass(nums)
    print(got, "OK" if sorted(got) == sorted(exp) else "SAI")

[2, 4] OK
[3] OK
[0] OK


**Độ phức tạp cách 3:** O(n) thời gian; bộ nhớ phụ O(1) ngoài danh sách đáp án. Vì điểm thay đổi đúng ±1 mỗi bước, hai chỉ số liền kề không thể cùng đạt max ⇒ đáp án dài tối đa `⌈(n+1)/2⌉` (ví dụ `[1,0,1,0,…]`).

### 7.3 Cách 4 — Hai lượt: tính max trước, rồi gom chỉ số

Tránh logic reset danh sách: lượt 1 tìm `mx`, lượt 2 gom mọi `i` có `score(i) = mx`.

In [12]:
def two_passes(nums):
    s = sum(nums); mx = s
    for x in nums:
        s += 1 - 2 * x                # 0 -> +1, 1 -> -1
        mx = max(mx, s)
    s = sum(nums); ans = [0] if s == mx else []
    for i, x in enumerate(nums, 1):
        s += 1 - 2 * x
        if s == mx: ans.append(i)
    return ans

for nums, exp in examples:
    got = two_passes(nums)
    print(got, "OK" if sorted(got) == sorted(exp) else "SAI")

[2, 4] OK
[3] OK
[0] OK


`1 − 2x` là mẹo gọn: `x = 0 → +1`, `x = 1 → −1`.

### 7.4 Cách 5 — Max prefix sum của `±1` bằng `accumulate`

In [13]:
def prefix_pm1(nums):
    W = list(accumulate((1 - 2 * x for x in nums), initial=0))     # W[i] = score(i) - sum(nums)
    mx = max(W)
    return [i for i, w in enumerate(W) if w == mx]

for nums, exp in examples:
    got = prefix_pm1(nums)
    print(got, "OK" if sorted(got) == sorted(exp) else "SAI")

[2, 4] OK
[3] OK
[0] OK


Không cần cộng `sum(nums)` vì hằng số không đổi vị trí max. Đây là phiên bản ngắn nhất và thường nhanh nhất trong Python (vòng lặp nằm trong `accumulate`/`max`).

## 8. TRACE từng bước

Trace cách 3 trên `nums = [1, 0, 0, 1, 0, 0, 1, 1]`.

In [14]:
def trace_one_pass(nums):
    score = sum(nums); mx = score; ans = [0]
    print(f"i=0: score={score} -> mx={mx}, ans={ans}")
    for i, x in enumerate(nums, 1):
        score += 1 if x == 0 else -1
        if score > mx:
            mx = score; ans = [i]; act = "MAX MỚI -> reset"
        elif score == mx:
            ans.append(i); act = "bằng max -> thêm"
        else:
            act = "nhỏ hơn -> bỏ"
        print(f"i={i}: nhảy qua {x}, score={score:>2} ({act:<17}) mx={mx}, ans={ans}")
    return ans

trace_one_pass([1, 0, 0, 1, 0, 0, 1, 1])

i=0: score=4 -> mx=4, ans=[0]
i=1: nhảy qua 1, score= 3 (nhỏ hơn -> bỏ    ) mx=4, ans=[0]
i=2: nhảy qua 0, score= 4 (bằng max -> thêm ) mx=4, ans=[0, 2]
i=3: nhảy qua 0, score= 5 (MAX MỚI -> reset ) mx=5, ans=[3]
i=4: nhảy qua 1, score= 4 (nhỏ hơn -> bỏ    ) mx=5, ans=[3]
i=5: nhảy qua 0, score= 5 (bằng max -> thêm ) mx=5, ans=[3, 5]
i=6: nhảy qua 0, score= 6 (MAX MỚI -> reset ) mx=6, ans=[6]
i=7: nhảy qua 1, score= 5 (nhỏ hơn -> bỏ    ) mx=6, ans=[6]
i=8: nhảy qua 1, score= 4 (nhỏ hơn -> bỏ    ) mx=6, ans=[6]


[6]

Bảng tóm tắt (`Σ nums = 4`):

| i | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|---|
| phần tử vừa qua | — | 1 | 0 | 0 | 1 | 0 | 0 | 1 | 1 |
| score | 4 | 3 | 4 | 5 | 4 | 5 | 6 | 5 | 4 |

Max 6 tại `i = 6` ⇒ đáp án `[6]`.

Trace cách 2 (in cả ba mảng tiền tố):

In [15]:
nums = [1, 0, 0, 1, 0, 0, 1, 1]
pre0 = list(accumulate((x ^ 1 for x in nums), initial=0)); pre1 = list(accumulate(nums, initial=0))
print("i     :", list(range(len(nums) + 1)))
print("pre0  :", pre0)
print("1 phải:", [pre1[-1] - p for p in pre1])
print("score :", [a + pre1[-1] - b for a, b in zip(pre0, pre1)])

i     : [0, 1, 2, 3, 4, 5, 6, 7, 8]
pre0  : [0, 0, 1, 2, 2, 3, 4, 4, 4]
1 phải: [4, 3, 3, 3, 2, 2, 2, 1, 0]
score : [4, 3, 4, 5, 4, 5, 6, 5, 4]


## 9. Code nộp LeetCode

In [16]:
from typing import List

class Solution:
    def maxScoreIndices(self, nums: List[int]) -> List[int]:
        score = sum(nums)               # i = 0: trái rỗng, phải = cả mảng
        best = score
        ans = [0]
        for i, x in enumerate(nums, 1): # x vừa chuyển từ phải sang trái
            if x == 0:
                score += 1
            else:
                score -= 1
            if score > best:
                best = score
                ans = [i]
            elif score == best:
                ans.append(i)
        return ans

In [17]:
sol = Solution()
for nums, exp in examples:
    got = sol.maxScoreIndices(nums)
    print(got, "| kỳ vọng (tùy thứ tự)", exp, "| OK" if sorted(got) == sorted(exp) else "| SAI")

[2, 4] | kỳ vọng (tùy thứ tự) [2, 4] | OK
[3] | kỳ vọng (tùy thứ tự) [3] | OK
[0] | kỳ vọng (tùy thứ tự) [0] | OK


## 10. Vì sao thuật toán đúng

**Công thức cập nhật.** `score(i + 1) − score(i)`: bên trái thêm `nums[i]` (đóng góp `[nums[i] = 0]`), bên phải bớt `nums[i]` (mất `[nums[i] = 1]`) ⇒ hiệu = `[nums[i] = 0] − [nums[i] = 1]` = `+1` hoặc `−1`. Với `score(0) = Σ nums` (số 1 trong cả mảng), quy nạp cho `score` đúng tại mọi `i`.

**Gom chỉ số max.** Bất biến (mục 6) đúng khi `i = 0` (`ans = [0]`, `mx = score(0)`). Bước `i`: nếu `score(i) > mx` thì mọi chỉ số cũ đều không còn là max ⇒ `ans = [i]`; nếu bằng ⇒ `i` cũng là max ⇒ thêm; nếu nhỏ hơn ⇒ không đổi. Sau `i = n`, `ans` là đúng tập các chỉ số đạt `max_{0..n} score`. ∎

**Hệ quả nhỏ:** vì `score` đổi đúng ±1 mỗi bước, hai chỉ số liên tiếp không thể cùng đạt max ⇒ `|ans| ≤ ⌈(n + 1)/2⌉`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space phụ |
|---|---|---|
| 1. Vét cạn | O(n²) | O(n) |
| 2. Tiền tố hai mảng | O(n) | O(n) |
| 3. Một lượt | O(n) | O(1) + đáp án |
| 4. Hai lượt | O(n) | O(1) + đáp án |
| 5. accumulate ±1 | O(n) | O(n) |

In [18]:
rng = random.Random(2155)
big = [rng.randint(0, 1) for _ in range(10**5)]
res = {}
for name, f in [("prefix_arrays", prefix_arrays), ("one_pass", one_pass), ("two_passes", two_passes),
                ("prefix_pm1", prefix_pm1), ("Solution", Solution().maxScoreIndices)]:
    t = time.perf_counter(); res[name] = f(big); dt = time.perf_counter() - t
    print(f"{name:<14} -> {len(res[name])} chỉ số, {dt*1000:.1f} ms")
assert len({tuple(sorted(v)) for v in res.values()}) == 1

prefix_arrays  -> 2 chỉ số, 17.3 ms
one_pass       -> 2 chỉ số, 6.3 ms
two_passes     -> 2 chỉ số, 8.4 ms
prefix_pm1     -> 2 chỉ số, 5.8 ms
Solution       -> 2 chỉ số, 6.2 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [19]:
ALL = {"brute": brute, "prefix_arrays": prefix_arrays, "one_pass": one_pass,
       "two_passes": two_passes, "prefix_pm1": prefix_pm1, "Solution": Solution().maxScoreIndices}
edge = examples + [
    ([0], [1]), ([1], [0]),
    ([0, 1], [1]),                 # điểm: 1, 2, 1
    ([1, 0], [0, 2]),              # điểm: 1, 0, 1
    ([1, 0, 1, 0], [0, 2, 4]),
    ([0, 1, 0, 1], [1, 3]),
    ([1] * 5 + [0] * 5, [0, 10]),
]
for nums, exp in edge:
    for name, f in ALL.items():
        a = list(nums)
        got = f(a)
        assert sorted(got) == sorted(exp), (name, nums, got, exp)
        assert a == nums
print("Edge case OK:", len(edge))

Edge case OK: 10


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

In [20]:
rng = random.Random(42)
for _ in range(3000):
    n = rng.randint(1, 30); p = rng.random()
    nums = [1 if rng.random() < p else 0 for _ in range(n)]
    ref = sorted(brute(nums))
    for name, f in ALL.items():
        got = f(list(nums))
        assert sorted(got) == ref, (name, nums)
        assert len(got) == len(set(got))
print("3000 case ngẫu nhiên OK")

3000 case ngẫu nhiên OK


### 12.3 Tính chất: không có hai chỉ số liên tiếp trong đáp án; đảo mảng + bù bit cho đáp án đối xứng

Đảo ngược mảng **và** đổi 0↔1: điểm chia `i` của mảng gốc ứng với điểm chia `n − i` của mảng mới với cùng điểm số.

In [21]:
rng = random.Random(7)
for _ in range(500):
    nums = [rng.randint(0, 1) for _ in range(rng.randint(1, 40))]; n = len(nums)
    ans = sorted(Solution().maxScoreIndices(nums))
    assert all(b - a >= 2 for a, b in zip(ans, ans[1:]))
    mirror = [1 - x for x in reversed(nums)]
    assert sorted(n - i for i in Solution().maxScoreIndices(mirror)) == ans
print("tính chất OK")

tính chất OK


### 12.4 Cực trị theo ràng buộc

- `n = 10⁵` toàn 0 ⇒ đáp án `[n]`; toàn 1 ⇒ `[0]`.
- Xen kẽ `1, 0, 1, 0, …` (n chẵn) ⇒ điểm dao động `n/2, n/2 − 1, n/2, …` ⇒ đáp án là mọi chỉ số chẵn — độ dài đáp án lớn nhất có thể.

In [22]:
N = 10**5
for name, f in ALL.items():
    if name == "brute": continue
    assert f([0] * N) == [N] and f([1] * N) == [0]
    alt = [1, 0] * (N // 2)
    assert sorted(f(alt)) == list(range(0, N + 1, 2))
print("cực trị OK; độ dài đáp án với mảng xen kẽ:", len(Solution().maxScoreIndices([1, 0] * (N // 2))))

cực trị OK; độ dài đáp án với mảng xen kẽ: 50001


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ duyệt `i = 0..n−1` (quên điểm chia `i = n`)

In [23]:
def wrong_miss_last(nums):
    n = len(nums)
    scores = [nums[:i].count(0) + nums[i:].count(1) for i in range(n)]    # SAI: thiếu i = n
    mx = max(scores)
    return [i for i, s in enumerate(scores) if s == mx]

print("bản sai :", wrong_miss_last([0, 0, 0]))
print("bản đúng:", Solution().maxScoreIndices([0, 0, 0]))

bản sai : [2]
bản đúng: [3]


**Vì sao sai:** có `n + 1` điểm chia (kể cả `i = n` khi bên phải rỗng). Với mảng toàn 0, max lại nằm đúng ở `i = n`. **Sửa:** `range(n + 1)`.

### Lỗi 2 — Quên điểm chia `i = 0` khi khởi tạo một lượt

In [24]:
def wrong_miss_first(nums):
    score = sum(nums); mx = -1; ans = []                 # SAI: không xét i = 0
    for i, x in enumerate(nums, 1):
        score += 1 if x == 0 else -1
        if score > mx: mx = score; ans = [i]
        elif score == mx: ans.append(i)
    return ans

print("bản sai :", wrong_miss_first([1, 1]))
print("bản đúng:", Solution().maxScoreIndices([1, 1]))

bản sai : [1]
bản đúng: [0]


**Sửa:** khởi tạo `mx = score(0) = Σ nums`, `ans = [0]`.

### Lỗi 3 — Gặp max mới nhưng **không reset** danh sách

In [25]:
def wrong_no_reset(nums):
    score = sum(nums); mx = score; ans = [0]
    for i, x in enumerate(nums, 1):
        score += 1 if x == 0 else -1
        if score >= mx:
            mx = score; ans.append(i)                     # SAI: max mới vẫn giữ chỉ số cũ
    return ans

print("bản sai :", wrong_no_reset([0, 0, 1, 0]))
print("bản đúng:", Solution().maxScoreIndices([0, 0, 1, 0]))

bản sai : [0, 1, 2, 4]
bản đúng: [2, 4]


**Sửa:** tách hai trường hợp: `>` ⇒ `ans = [i]`; `==` ⇒ `append`.

### Lỗi 4 — Cập nhật điểm sai chiều (`+1` cho số 1)

In [26]:
def wrong_direction(nums):
    score = sum(nums); mx = score; ans = [0]
    for i, x in enumerate(nums, 1):
        score += 1 if x == 1 else -1                       # SAI: đảo dấu
        if score > mx: mx = score; ans = [i]
        elif score == mx: ans.append(i)
    return ans

print("bản sai :", wrong_direction([0, 0, 1, 0]))
print("bản đúng:", Solution().maxScoreIndices([0, 0, 1, 0]))

bản sai : [0]
bản đúng: [2, 4]


**Vì sao sai:** số 0 chuyển sang trái thì **được** tính (trái đếm số 0) ⇒ `+1`; số 1 chuyển sang trái thì **mất** (phải đếm số 1) ⇒ `−1`. **Sửa:** `+1` cho 0, `−1` cho 1 (`1 − 2x`).

### Lỗi 5 — Lệch chỉ số khi dùng `enumerate` mặc định (bắt đầu 0)

In [27]:
def wrong_index_base(nums):
    score = sum(nums); mx = score; ans = [0]
    for i, x in enumerate(nums):                           # SAI: i là chỉ số phần tử, không phải điểm chia
        score += 1 if x == 0 else -1
        if score > mx: mx = score; ans = [i]
        elif score == mx: ans.append(i)
    return ans

print("bản sai :", wrong_index_base([0, 0, 1, 0]))
print("bản đúng:", Solution().maxScoreIndices([0, 0, 1, 0]))

bản sai : [1, 3]
bản đúng: [2, 4]


**Sửa:** sau khi xử lý `nums[i]`, điểm chia mới là `i + 1` ⇒ `enumerate(nums, 1)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chia mảng tại mọi vị trí, điểm = (đếm bên trái) + (đếm bên phải)" ⇒ dịch vách ngăn từng bước, cập nhật **O(1)** (hoặc tiền tố/hậu tố).
- Luôn nhớ có **n + 1** điểm chia khi cho phép phần rỗng.
- "Trả về mọi chỉ số đạt max" ⇒ pattern **reset khi lớn hơn, append khi bằng** (hoặc hai lượt).
- Hằng số cộng thêm không đổi vị trí max ⇒ quy về **max prefix sum** của `±1`.
- Bài cùng họ: 1422 Maximum Score After Splitting a String (giống hệt nhưng cả hai phần không rỗng), 2483 Minimum Penalty for a Shop, 1525 Number of Good Ways to Split a String, 2270.

## 15. Biến thể của bài

### 15.1 Biến thể A — LC 1422: chuỗi `'0'/'1'`, **cả hai phần không rỗng**, trả về **điểm** lớn nhất

In [28]:
def max_score_split(s):
    score = s.count("1"); best = -1
    for i in range(1, len(s)):                 # điểm chia 1..n-1 (không rỗng)
        score += 1 if s[i - 1] == "0" else -1
        best = max(best, score)
    return best

def max_score_split_brute(s):
    return max(s[:i].count("0") + s[i:].count("1") for i in range(1, len(s)))

print(max_score_split("011101"), max_score_split("00111"), max_score_split("1111"))
rng = random.Random(151)
for _ in range(500):
    s = "".join(rng.choice("01") for _ in range(rng.randint(2, 15)))
    assert max_score_split(s) == max_score_split_brute(s)
print("khớp brute force")

5 5 3
khớp brute force


### 15.2 Biến thể B — LC 2483: cửa hàng đóng cửa ở giờ `j`, phạt = số `'N'` trước `j` + số `'Y'` từ `j` trở đi; tìm `j` **nhỏ nhất** có phạt **nhỏ nhất**

Cùng cơ chế vách ngăn (đổi max thành min, chỉ giữ chỉ số đầu tiên).

In [29]:
def best_closing_time(customers):
    pen = customers.count("Y"); best = pen; ans = 0
    for j, c in enumerate(customers, 1):
        pen += 1 if c == "N" else -1
        if pen < best: best = pen; ans = j
    return ans

def best_closing_brute(customers):
    pens = [customers[:j].count("N") + customers[j:].count("Y") for j in range(len(customers) + 1)]
    return pens.index(min(pens))

print(best_closing_time("YYNY"), best_closing_time("NNNNN"), best_closing_time("YYYY"))
rng = random.Random(152)
for _ in range(500):
    s = "".join(rng.choice("YN") for _ in range(rng.randint(1, 15)))
    assert best_closing_time(s) == best_closing_brute(s)
print("khớp brute force")

2 0 4
khớp brute force


### 15.3 Biến thể C — Trọng số: bên trái mỗi số 0 được `a` điểm, bên phải mỗi số 1 được `b` điểm

Cập nhật khi vách nhảy qua `x`: `+a` nếu `x = 0`, `−b` nếu `x = 1`. Khởi tạo `score(0) = b·Σnums`.

In [30]:
def weighted_indices(nums, a, b):
    score = b * sum(nums); mx = score; ans = [0]
    for i, x in enumerate(nums, 1):
        score += a if x == 0 else -b
        if score > mx: mx = score; ans = [i]
        elif score == mx: ans.append(i)
    return ans

def weighted_brute(nums, a, b):
    sc = [a * nums[:i].count(0) + b * nums[i:].count(1) for i in range(len(nums) + 1)]
    return [i for i, v in enumerate(sc) if v == max(sc)]

print(weighted_indices([0, 0, 1, 0], 1, 1), weighted_indices([0, 0, 1, 0], 1, 3), weighted_indices([0, 0, 1, 0], 3, 1))
rng = random.Random(153)
for _ in range(500):
    nums = [rng.randint(0, 1) for _ in range(rng.randint(1, 12))]; a, b = rng.randint(1, 4), rng.randint(1, 4)
    assert weighted_indices(nums, a, b) == weighted_brute(nums, a, b)
print("khớp brute force")

[2, 4] [2] [4]
khớp brute force


Với trọng số, hai chỉ số liền kề **có thể** cùng max? Không — vì mỗi bước đổi `+a` hoặc `−b` (khác 0). Nhưng nếu cho phép `a = 0` hoặc `b = 0` thì có thể.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | đếm lại mỗi điểm chia | O(n²) | O(n) | kiểm chứng |
| 2. Tiền tố hai mảng | `pre0[i] + (tổng1 − pre1[i])` | O(n) | O(n) | dễ kiểm tra |
| 3. Một lượt | ±1, reset/append | O(n) | O(1) | **mặc định** |
| 4. Hai lượt | tìm max rồi gom | O(n) | O(1) | tránh logic reset |
| 5. accumulate ±1 | max prefix sum | O(n) | O(n) | ngắn, nhanh trong Python |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `maxScoreIndices([1,0,1,0,1])` và `maxScoreIndices([0,1,1,0,0,1,1])`?

**Bài 2 (tính tay):** Tính dãy điểm `score(0..n)` cho `[1, 1, 0, 0, 0, 1]`.

**Bài 3 (điền TODO):** Viết `bt3_todo(nums)` trả về **(điểm cao nhất, chỉ số nhỏ nhất đạt được)**.

**Bài 4 (tự viết lại):** Viết `bt4_todo(nums)` cài cách 5 (max prefix sum của ±1) không nhìn code.

**Bài 5 (mini challenge):** Có `q` truy vấn `(l, r)`: xét mảng con `nums[l..r]` như một mảng độc lập; trả về **điểm cao nhất** cho mỗi truy vấn. Viết `bt5_todo(nums, queries)` với O(độ dài đoạn) mỗi truy vấn, dùng tiền tố chung.

In [31]:
def bt3_todo(nums):
    # TODO: (điểm cao nhất, chỉ số nhỏ nhất)
    pass

def bt4_todo(nums):
    # TODO: max prefix sum của ±1
    pass

def bt5_todo(nums, queries):
    # TODO: điểm cao nhất cho mỗi mảng con nums[l..r]
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `score(0) = Σ nums`. Mức 2: `[1,0,1,0,1]`: 3, 2, 3, 2, 3, 2. Mức 3: `[0, 2, 4]`; mảng hai: điểm 4, 5, 4, 3, 4, 5, 4, 3 ⇒ `[1, 5]`.

**Bài 2.** Mức 1: `Σ = 3`. Mức 2: qua 1: −1, qua 0: +1. Mức 3: `[3, 2, 1, 2, 3, 4, 3]`.

**Bài 3.** Mức 1: một lượt. Mức 2: chỉ cập nhật chỉ số khi **lớn hơn hẳn**. Mức 3: trả `(best, best_i)`.

**Bài 4.** Mức 1: `W = accumulate(1 − 2x, initial=0)`. Mức 2: `mx = max(W)`. Mức 3: các `i` có `W[i] == mx`.

**Bài 5.** Mức 1: điểm của mảng con tại điểm chia `l ≤ i ≤ r + 1` = (#0 trong `nums[l..i−1]`) + (#1 trong `nums[i..r]`). Mức 2: dùng `pre0`, `pre1` toàn cục. Mức 3: `pre0[i] − pre0[l] + pre1[r+1] − pre1[i]`, lấy max theo `i`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `[0, 2, 4]`, `[1, 5]`. **Bài 2:** `[3, 2, 1, 2, 3, 4, 3]`.

In [32]:
assert sorted(Solution().maxScoreIndices([1, 0, 1, 0, 1])) == [0, 2, 4]
assert sorted(Solution().maxScoreIndices([0, 1, 1, 0, 0, 1, 1])) == [1, 5]
nums = [1, 1, 0, 0, 0, 1]
assert [score(nums, i) for i in range(len(nums) + 1)] == [3, 2, 1, 2, 3, 4, 3]

def bt3(nums):
    s = sum(nums); best, bi = s, 0
    for i, x in enumerate(nums, 1):
        s += 1 - 2 * x
        if s > best: best, bi = s, i
    return best, bi

def bt4(nums):
    W = list(accumulate((1 - 2 * x for x in nums), initial=0)); mx = max(W)
    return [i for i, w in enumerate(W) if w == mx]

def bt5(nums, queries):
    pre0 = list(accumulate((x ^ 1 for x in nums), initial=0)); pre1 = list(accumulate(nums, initial=0))
    return [max(pre0[i] - pre0[l] + pre1[r + 1] - pre1[i] for i in range(l, r + 2)) for l, r in queries]

rng = random.Random(1955)
for _ in range(1500):
    nums = [rng.randint(0, 1) for _ in range(rng.randint(1, 25))]
    sc = [score(nums, i) for i in range(len(nums) + 1)]
    assert bt3(nums) == (max(sc), sc.index(max(sc)))
    assert bt4(nums) == sorted(brute(nums))
    qs = []
    for _ in range(3):
        l = rng.randrange(len(nums)); r = rng.randrange(l, len(nums)); qs.append((l, r))
    assert bt5(nums, qs) == [max(score(nums[l:r + 1], i) for i in range(r - l + 2)) for l, r in qs]
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
score = sum(nums); best = score; ans = [0]       # điểm chia i = 0
for i, x in enumerate(nums, 1):                  # điểm chia i sau khi qua nums[i-1]
    score += 1 if x == 0 else -1
    if score > best: best, ans = score, [i]
    elif score == best: ans.append(i)
return ans
```

**Checklist:**
- [ ] Xét đủ **n + 1** điểm chia (`i = 0` và `i = n`).
- [ ] `+1` khi qua số 0, `−1` khi qua số 1.
- [ ] Max mới ⇒ **reset** danh sách; bằng ⇒ thêm.
- [ ] Chỉ số điểm chia là `i` trong `enumerate(nums, 1)`.
- [ ] Đã thử: toàn 0 (đáp án `[n]`), toàn 1 (đáp án `[0]`), xen kẽ.